# SGLang vs vLLM —— 同硬件同模型受控对照

在**同一张 Colab T4**、**同一个 Qwen2.5-0.5B-Instruct**、**同一套压测脚本**下比较两个推理框架。
两边都暴露 OpenAI 兼容的 `/v1/chat/completions`，因此客户端一行不用改，
差异可完全归因于服务端实现。

| | vLLM | SGLang |
|---|---|---|
| 前缀复用机制 | `--enable-prefix-caching` | **RadixAttention**（默认开，`--disable-radix-cache` 关） |
| 批处理 | Continuous Batching | Continuous Batching |

**四组配置**：{vLLM, SGLang} × {前缀复用 开, 关}。
指标：吞吐 tok/s、TTFT p50/p99、TPOT p50。

> ⚠️ SGLang 与 vLLM 的依赖（torch / flashinfer）会互相覆盖，
> **本 notebook 必须在独立的 Colab 会话里跑**，不要和 vLLM 那份共用运行时。


## 0. 环境


In [ ]:
# 只用 nvidia-smi，不 import torch —— 本 notebook 主进程全程不加载 torch/sglang。
import subprocess
out = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,compute_cap",
                      "--format=csv"],capture_output=True,text=True).stdout
print(out)
name = out.strip().splitlines()[-1].split(",")[0].strip()
print("Tensor Core:", "无（GTX 16 系）" if "GTX 16" in name else "有")


## 1. 装 SGLang（几分钟，不需要重启）

**沿用 vLLM 那份 notebook 验证过的做法：主进程不 `import torch`、不 `import sglang`。**

Colab 预装 cu12 的 torch，新框架会把它升到 cu13，但**已加载进当前进程的动态库不会重新解析**，
于是 `import sglang` 报 `libcudart.so.13`，只能靠重启内核绕 —— 而重启会打断「全部运行」。
服务用 `subprocess` 起、压测用 `!python` 跑，都是新进程，本来就能正确解析，
所以主进程不碰它们，重启环节就不必存在。

torchvision / torchaudio 与 sglang 放同一条 pip 命令解析，避免 CUDA 标签不一致；
**不要用卸载来绕** —— 框架内部会 import torchvision，卸掉会崩在更深的地方。


In [ ]:
import importlib.metadata as md, subprocess, sys

PKGS = ["sglang[all]", "aiohttp", "torchvision", "torchaudio"]
CHECK = ["sglang", "aiohttp", "torchvision", "torchaudio"]

def ver(p):
    try:    return md.version(p)
    except Exception: return None

missing = [p for p in CHECK if ver(p) is None]
if missing:
    print("缺失：", ", ".join(missing), "→ 一并解析安装（几分钟）")
    subprocess.run([sys.executable,"-m","pip","install","-q",*PKGS], check=False)
else:
    print("依赖齐备，跳过安装")

for p in CHECK:
    print(f"  {p:<12} {ver(p)}")

r = subprocess.run([sys.executable,"-c","import sglang,torch;print(sglang.__version__,torch.__version__)"],
                   capture_output=True, text=True)
print("子进程内 sglang / torch：", (r.stdout or r.stderr).strip()[:200])


## 2. 写出压测脚本（与 vLLM 那轮完全同一份）


In [ ]:
import io
files = {}

files['bench_serving.py'] = r'''# -*- coding: utf-8 -*-
"""vLLM 服务端压测：并发扫描下的吞吐 / TTFT / TPOT，以及前缀复用的效果。

指标定义（与 JD 里那套一致）：
  TTFT  Time To First Token   —— 首 token 延迟，决定交互体感
  TPOT  Time Per Output Token —— 首 token 之后的平均出词间隔
  吞吐   总输出 token 数 / 墙钟时间

用法（先 bash serve.sh 起服务）：
  python bench_serving.py                 # 并发扫描
  python bench_serving.py --prefix-test   # 前缀复用对照
"""
import argparse, asyncio, json, statistics as st, time
import aiohttp

URL = "http://127.0.0.1:8000/v1/chat/completions"
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

# 一段较长的共享 system prompt：开 --enable-prefix-caching 后其 prefill 只算一次
SHARED_PREFIX = (
    "You are a meticulous technical assistant. Answer concisely and precisely. "
    "Always reason step by step before answering. " * 20
)


async def one_request(sess, prompt, max_tokens, use_prefix):
    msgs = ([{"role": "system", "content": SHARED_PREFIX}] if use_prefix else []) + \
           [{"role": "user", "content": prompt}]
    body = {"model": MODEL, "messages": msgs, "max_tokens": max_tokens,
            "temperature": 0.0, "stream": True}
    t0 = time.perf_counter()
    ttft, n_tok, last = None, 0, t0
    async with sess.post(URL, json=body) as resp:
        async for raw in resp.content:
            line = raw.decode("utf-8").strip()
            if not line.startswith("data: ") or line == "data: [DONE]":
                continue
            delta = json.loads(line[6:])["choices"][0].get("delta", {})
            if delta.get("content"):
                now = time.perf_counter()
                if ttft is None:
                    ttft = now - t0
                n_tok += 1
                last = now
    return dict(ttft=ttft or 0.0, total=last - t0, n_tok=n_tok)


async def run_batch(n_conc, n_req, max_tokens, use_prefix):
    prompts = [f"Explain concept #{i} in distributed systems." for i in range(n_req)]
    sem = asyncio.Semaphore(n_conc)

    async def guarded(sess, p):
        async with sem:
            return await one_request(sess, p, max_tokens, use_prefix)

    timeout = aiohttp.ClientTimeout(total=600)
    async with aiohttp.ClientSession(timeout=timeout) as sess:
        await one_request(sess, "warmup", 4, use_prefix)          # 预热
        t0 = time.perf_counter()
        rs = await asyncio.gather(*(guarded(sess, p) for p in prompts))
        wall = time.perf_counter() - t0

    tot_tok = sum(r["n_tok"] for r in rs)
    tpots = [(r["total"] - r["ttft"]) / max(r["n_tok"] - 1, 1) for r in rs if r["n_tok"] > 1]
    return dict(conc=n_conc, wall=wall, tput=tot_tok / wall, rps=len(rs) / wall,
                ttft_p50=st.median(r["ttft"] for r in rs),
                ttft_p99=sorted(r["ttft"] for r in rs)[int(len(rs) * 0.99) - 1],
                tpot_p50=st.median(tpots) if tpots else 0.0, tot_tok=tot_tok)


async def sweep(args):
    print(f"{'并发':>5}{'请求':>6}{'墙钟s':>9}{'吞吐 tok/s':>13}{'RPS':>8}"
          f"{'TTFT p50':>11}{'TTFT p99':>11}{'TPOT p50':>11}")
    print("-" * 74)
    out = []
    for c in [1, 2, 4, 8, 16, 32]:
        r = await run_batch(c, max(c * 4, 16), args.max_tokens, use_prefix=False)
        print(f"{r['conc']:>5}{max(c*4,16):>6}{r['wall']:>9.2f}{r['tput']:>13.1f}"
              f"{r['rps']:>8.2f}{r['ttft_p50']*1e3:>10.1f}ms{r['ttft_p99']*1e3:>10.1f}ms"
              f"{r['tpot_p50']*1e3:>10.2f}ms")
        out.append(r)
    json.dump(out, open("sweep_results.json", "w"), indent=1)
    base = out[0]["tput"]
    print(f"\ncontinuous batching 收益：并发 1 → 32，吞吐 "
          f"{base:.1f} → {out[-1]['tput']:.1f} tok/s（{out[-1]['tput']/base:.1f}×），"
          f"TTFT p50 {out[0]['ttft_p50']*1e3:.0f} → {out[-1]['ttft_p50']*1e3:.0f} ms")
    print("吞吐与延迟的取舍就在这张表里：并发拉高吞吐涨，但 TTFT 同步恶化。")


async def prefix_test(args):
    print("前缀复用对照（服务端需带 --enable-prefix-caching 启动）")
    print(f"{'场景':<26}{'吞吐 tok/s':>13}{'TTFT p50':>12}")
    print("-" * 51)
    for label, up in [("无共享前缀", False), (f"共享前缀 ({len(SHARED_PREFIX)} 字符)", True)]:
        r = await run_batch(8, 32, args.max_tokens, use_prefix=up)
        print(f"{label:<26}{r['tput']:>13.1f}{r['ttft_p50']*1e3:>11.1f}ms")
    print("\n共享前缀命中 KV cache 后，重复的 prefill 不再重算，TTFT 应显著下降。")
    print("对比未开 --enable-prefix-caching 重启服务再跑一次，差值即为该特性的真实收益。")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--max-tokens", type=int, default=128)
    ap.add_argument("--prefix-test", action="store_true")
    a = ap.parse_args()
    asyncio.run(prefix_test(a) if a.prefix_test else sweep(a))
'''

files['isolate.py'] = r'''# -*- coding: utf-8 -*-
"""隔离测试：一次性发 N 条并发请求，期间无新请求到达。
用于区分「稳态 batch=N 解码慢」与「新请求 prefill 插队拖累解码」。"""
import asyncio, sys, time, json, statistics as st
import aiohttp
URL="http://127.0.0.1:8000/v1/chat/completions"; MODEL="Qwen/Qwen2.5-0.5B-Instruct"

async def one(s, i, n_tok):
    b={"model":MODEL,"messages":[{"role":"user","content":f"Explain idea {i} briefly."}],
       "max_tokens":n_tok,"temperature":0.0,"stream":True}
    t0=time.perf_counter(); ttft=None; n=0; last=t0
    async with s.post(URL,json=b) as r:
        async for raw in r.content:
            l=raw.decode().strip()
            if not l.startswith("data: ") or l=="data: [DONE]": continue
            d=json.loads(l[6:])["choices"][0].get("delta",{})
            if d.get("content"):
                now=time.perf_counter()
                if ttft is None: ttft=now-t0
                n+=1; last=now
    return (ttft or 0), last-t0, n

async def burst(n, n_tok=64):
    """严格同时发 n 条，全部跑完才结束 —— 稳态就是 batch=n。"""
    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=600)) as s:
        await one(s,-1,4)                       # 预热
        t0=time.perf_counter()
        rs=await asyncio.gather(*(one(s,i,n_tok) for i in range(n)))
        wall=time.perf_counter()-t0
    tp=[(tot-tt)/max(k-1,1) for tt,tot,k in rs if k>1]
    tot=sum(k for _,_,k in rs)
    print(f"  一次性 {n:>2} 条并发: 墙钟 {wall:>6.2f}s  总吞吐 {tot/wall:>6.1f} tok/s  "
          f"TPOT p50 {st.median(tp)*1e3:>7.2f} ms")

async def main():
    print("=== 无新到达的纯稳态测试 ===")
    for n in [1, 2, 3, 4, 8, 16]:
        await burst(n)

asyncio.run(main())
'''

for n,x in files.items():
    io.open(n,"w",encoding="utf-8").write(x); print(" ", n, len(x.splitlines()), "行")



## 3. 启动器


In [ ]:
import subprocess, time, requests

def serve_sglang(extra, tag):
    subprocess.run(["pkill","-f","sglang.launch_server"],check=False); time.sleep(8)
    cmd = [__import__("sys").executable,"-m","sglang.launch_server",
           "--model-path","Qwen/Qwen2.5-0.5B-Instruct",
           "--host","127.0.0.1","--port","8000",
           "--dtype","half","--context-length","2048",
           "--mem-fraction-static","0.85"] + extra
    log = open(f"/content/sgl_{tag}.log","w")
    p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    for i in range(240):
        try:
            if requests.get("http://127.0.0.1:8000/v1/models",timeout=2).status_code==200:
                print(f"[{tag}] 就绪，用时 {i*2}s"); return p
        except Exception: pass
        time.sleep(2)
    print(f"[{tag}] 启动失败，日志尾部：")
    print(open(f"/content/sgl_{tag}.log").read()[-2000:]); return None


## 4. SGLang · RadixAttention 开（默认）

SGLang 的前缀复用叫 RadixAttention，默认开启，无需额外参数。


In [ ]:
proc = serve_sglang([], "radix_on")


In [ ]:
!python -u isolate.py


In [ ]:
!python -u bench_serving.py


In [ ]:
!python -u bench_serving.py --prefix-test


## 5. SGLang · RadixAttention 关（受控对照）


In [ ]:
proc = serve_sglang(["--disable-radix-cache"], "radix_off")


In [ ]:
!python -u bench_serving.py --prefix-test


## 6. 怎么读

把四组数并排：

| | 前缀复用 开 | 前缀复用 关 | 差值 |
|---|---|---|---|
| vLLM（`--enable-prefix-caching`） | | | |
| SGLang（RadixAttention） | | | |

两个维度都值得看：

1. **横向**：各自开/关的差值 = 该框架前缀复用的真实收益。
2. **纵向**：同一开关状态下两框架的绝对值 = 框架实现差异。

**如实记录不利结果。** 若某一框架无收益或反而更慢，照写，并说明可能原因
（0.5B 模型 prefill 本就便宜、共享前缀仅 2380 字符、T4 上 prefill 不是瓶颈等）。
